# LeNet5

* 这个文件用以帮助理解CNN
* 这是一个经典的教学案例，利用LeNet5学习MINST数据集

## 卷积层（Convolutional Layer）

卷积层是 CNN 的核心，用于从输入图像中提取局部特征，例如边缘、纹理、角点等。

- **基本思想**：使用若干个可学习的卷积核（filter）在图像上滑动，对局部区域进行加权计算。
- **作用**：自动提取特征，并保留空间结构信息。
- **特点**：
    - 局部连接：每个神经元只关注输入的一小块区域。
    - 参数共享：同一个卷积核在整张图像上重复使用，减少参数量。
- **输出**：通常得到多个特征图（feature maps）。

> 可以理解为：卷积层负责“发现图像中有什么特征”。


## 池化层（Pooling Layer）

池化层通常接在卷积层后面，用于对特征图进行降采样。

- **基本思想**：在局部区域内取一个代表值。
- **常见方式**：
    - 最大池化（Max Pooling）：取区域中的最大值
    - 平均池化（Average Pooling）：取区域中的平均值
- **作用**：
    - 减少特征图尺寸，降低计算量
    - 保留主要特征，抑制噪声
    - 提高模型对平移和局部变化的鲁棒性

> 可以理解为：池化层负责“压缩信息，保留重点”。


## 全连接层（Fully Connected Layer）

全连接层通常位于网络后部，用于根据前面提取到的特征完成分类或回归任务。

- **基本思想**：将前面提取到的特征展开成一维向量，并与每个神经元全部连接。
- **作用**：
    - 综合高层特征
    - 输出最终预测结果
- **在分类任务中**：
    - 最后一层通常输出每个类别的得分
    - 再通过 Softmax 转换为各类别的概率

> 可以理解为：全连接层负责“根据提取到的特征进行最终判断”。


## 总结

在 CNN 中：

1. **卷积层**：提取特征  
2. **池化层**：压缩特征  
3. **全连接层**：输出结果  

因此，CNN 可以从原始图像中逐步学习到从低级到高级的特征表示。

In [57]:
import torchvision
import torchvision.transforms as transforms
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [58]:
# load and preprocess the MNIST dataset
transform = transforms.Compose([transforms.ToTensor()])

trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
testset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)

In [59]:
# 定义LeNet-5模型
class LeNet5(nn.Module):
    def __init__(self):
        super(LeNet5, self).__init__()
        # 第一卷积层，3x3卷积核，输入通道1，输出通道5
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5)
        self.conv1.activation = nn.ReLU()
        # 第一池化层，平均池化
        self.pool1 = nn.AvgPool2d(2)
        # 第二卷积层，3x3卷积核，输入通道5，输出通道16
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        self.conv2.activation = nn.ReLU()
        # 第二池化层，平均池化
        self.pool2 = nn.AvgPool2d(2)
        # 全连接层
        self.fc1 = nn.Linear(16 * 4 * 4, 120)
        self.fc1.activation = nn.ReLU()
        self.fc2 = nn.Linear(120, 84)
        self.fc2.activation = nn.ReLU()
        self.fc3 = nn.Linear(84, 10)  # 输出层，10类

    def forward(self, x):
        x = self.pool1(self.conv1.activation(self.conv1(x)))
        x = self.pool2(self.conv2.activation(self.conv2(x)))
        x = x.view(-1, 16 * 4 * 4)  # 展平，view(-1, 16 * 4 * 4)表示将输入调整为(batch_size, 16*4*4)
        x = self.fc1(x)
        x = self.fc2(x)
        x = self.fc3(x)
        return x

In [60]:
# 构建模型
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LeNet5().to(device)

In [61]:
# 损失函数和优化器
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [62]:
# tranning

epochs = 5

for epoch in range(epochs):
    model.train()
    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()  # 清除之前的梯度
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        loss.backward()  # 反向传播计算梯度
        optimizer.step()  # 更新模型参数

        # 打印loss
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

Epoch [1/5], Loss: 2.3025
Epoch [1/5], Loss: 2.2946
Epoch [1/5], Loss: 2.2990
Epoch [1/5], Loss: 2.3061
Epoch [1/5], Loss: 2.2929
Epoch [1/5], Loss: 2.2984
Epoch [1/5], Loss: 2.2907
Epoch [1/5], Loss: 2.2573
Epoch [1/5], Loss: 2.2662
Epoch [1/5], Loss: 2.2576
Epoch [1/5], Loss: 2.2688
Epoch [1/5], Loss: 2.2601
Epoch [1/5], Loss: 2.2200
Epoch [1/5], Loss: 2.2432
Epoch [1/5], Loss: 2.2233
Epoch [1/5], Loss: 2.1927
Epoch [1/5], Loss: 2.1645
Epoch [1/5], Loss: 2.1836
Epoch [1/5], Loss: 2.1336
Epoch [1/5], Loss: 2.0686
Epoch [1/5], Loss: 2.0183
Epoch [1/5], Loss: 1.9792
Epoch [1/5], Loss: 1.9371
Epoch [1/5], Loss: 1.9186
Epoch [1/5], Loss: 1.8845
Epoch [1/5], Loss: 1.6797
Epoch [1/5], Loss: 1.7994
Epoch [1/5], Loss: 1.6515
Epoch [1/5], Loss: 1.6755
Epoch [1/5], Loss: 1.6709
Epoch [1/5], Loss: 1.3735
Epoch [1/5], Loss: 1.3897
Epoch [1/5], Loss: 1.4929
Epoch [1/5], Loss: 1.3489
Epoch [1/5], Loss: 1.2282
Epoch [1/5], Loss: 1.3461
Epoch [1/5], Loss: 1.1385
Epoch [1/5], Loss: 1.2870
Epoch [1/5],

In [63]:
correct = 0
total = 0

with torch.no_grad():

    for images, labels in testloader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

print("Accuracy:", 100 * correct / total)

Accuracy: 98.74


In [72]:
# prediction

model.eval()
print(F.softmax(model((torch.randn(1, 1, 28, 28).to(device))), dim=1))  # 测试模型输出，利用softmax将输出转换为概率分布
with torch.no_grad():
    for images, labels in testloader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images[:5])

        _, predicted = torch.max(outputs, 1)    # 1表示第一维度

        print("Predicted:", predicted.cpu().numpy())
        print("Actual:", labels[:5].cpu().numpy())
        break  # 只显示第一批次的预测结果

tensor([[2.9135e-11, 5.5203e-03, 2.5110e-06, 1.4358e-09, 9.9447e-01, 5.1178e-13,
         4.3025e-09, 7.0911e-06, 7.3463e-10, 1.1917e-11]], device='cuda:0',
       grad_fn=<SoftmaxBackward0>)
Predicted: [7 2 1 0 4]
Actual: [7 2 1 0 4]
